# 02 — Exploratory Data Analysis & Visualization

Understand PBX decline patterns across countries, visualize trends, and identify key covariates.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.data.preprocessor import add_pstn_phaseout_feature
from src.data.fetcher import load_config

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)

In [ ]:
panel = pd.read_csv('data/processed/panel_data.csv')
config = load_config()
print(f"Panel loaded: {panel.shape[0]} rows, {panel.shape[1]} cols")
print(f"\nColumns:\n{panel.columns.tolist()}")
print(f"\nCountries: {panel['country'].nunique()}")
print(f"Years: {panel['year'].min()}-{panel['year'].max()}")

In [ ]:
panel.describe()

## 2.1 PBX Decline Trends by Country

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
titles = ['Asia-Pacific', 'Europe', 'Americas']
regions = ['asia_pacific', 'europe', 'americas']
colors = plt.cm.Set2(np.linspace(0, 1, 12))

for idx, (region_key, title) in enumerate(zip(regions, titles)):
    ax = axes[idx]
    codes = []
    for item in config['countries'][region_key]:
        for code in item:
            codes.append(code)
    region_data = panel[panel['country'].isin(codes)]
    
    for i, code in enumerate(codes):
        ctry = region_data[region_data['country'] == code].sort_values('year')
        ax.plot(ctry['year'], ctry['fixed_subs_value'], 
                marker='o', label=code.upper(), color=colors[i])
    
    ax.set_title(title)
    ax.set_xlabel('Year')
    ax.set_ylabel('Fixed Telephone Subscriptions (per 100)')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('data/processed/trends_by_region.png', dpi=150, bbox_inches='tight')
plt.show()

## 2.2 Broadband Penetration vs Fixed Line Decline

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
for country in panel['country'].unique()[:6]:
    ctry = panel[panel['country'] == country].sort_values('year')
    ax.scatter(ctry['broadband_value'], ctry['fixed_subs_value'],
               marker='o', label=country.upper())
ax.set_xlabel('Broadband Penetration (per 100)')
ax.set_ylabel('Fixed Line Penetration (per 100)')
ax.set_title('Broadband vs Fixed Line (substitution effect)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2.3 PSTN Phaseout Status Map

In [ ]:
phaseout = panel.groupby('country')['has_pstn_phaseout'].max().reset_index()
phaseout_counts = phaseout['has_pstn_phaseout'].value_counts()
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(['No phaseout policy', 'Has phaseout policy'], phaseout_counts.values,
       color=['lightsalmon', 'lightgreen'], edgecolor='gray')
ax.set_title('PSTN Phaseout Policy by Country')
ax.set_ylabel('Number of countries')
for i, v in enumerate(phaseout_counts.values):
    ax.text(i, v + 0.1, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

print("\nCountries WITH PSTN phaseout policy:")
for _, row in phaseout[phaseout['has_pstn_phaseout'] == 1].iterrows():
    print(f"  - {row['country'].upper()}")
print("\nCountries WITHOUT PSTN phaseout policy:")
for _, row in phaseout[phaseout['has_pstn_phaseout'] == 0].iterrows():
    print(f"  - {row['country'].upper()}")

**EN — PSTN phaseout map.** Countries with an announced PSTN/copper switch-off (a policy commitment to retire the analog network) face structurally faster legacy decline. Treat this as a binary risk flag when prioritising markets.

**繁中 — PSTN 退場地圖。** 已宣布 PSTN/銅纜停用（即承諾淘汰類比網路之政策）的國家，其傳統線路衰退在結構上更快。排序市場時，可將此視為二元風險旗標。

## 2.4 Correlation Matrix

In [ ]:
corr_cols = ['fixed_subs_value', 'broadband_value', 'gdp_per_capita_value', 
             'urban_pop_value', 'has_pstn_phaseout']
corr_available = [c for c in corr_cols if c in panel.columns]
corr = panel[corr_available].corr()

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('Correlation Matrix of Market Covariates')
plt.tight_layout()
plt.savefig('data/processed/correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nKey insight: Strong negative corr between broadband and fixed line = substitution effect.")

**EN — Correlation matrix.** A strong *negative* correlation between broadband and fixed-line penetration is the substitution signal: as broadband rises, fixed lines (and the PBX they feed) fall. Correlation is not causation, but it motivates broadband as a survival covariate in notebook 04.

**繁中 — 相關矩陣。** 寬頻與固網滲透率呈強烈*負*相關，即替代訊號：寬頻上升、固網（及其所承載的 PBX）下降。相關不等於因果，但這支持在筆記本 04 將寬頻作為存活共變量。